# Week 76, Day 5: Your own HTTP server

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w076d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('netsim.py', 'w', encoding='utf-8') as f:
    f.write('"""Pretend TCP connections that run in the page, for practising sockets (Week 76).\n\na, b = socketpair()          two connected ends: bytes sent on one arrive at the other\na.send(data) -> int          sends SOME of data (perhaps not all), returning how many bytes\na.sendall(data)              (not provided: that\'s an exercise)\nb.recv(n) -> bytes           returns between 1 and n bytes, or b"" once the other end has\n                             closed and everything has been read\na.shutdown()                 says "I\'ve finished sending" (like sock.shutdown(socket.SHUT_WR)):\n                             the other end\'s recv then returns b"", but this end can still recv\na.close()                    closes this end completely\n\nLike real TCP, the connection is a stream of bytes with no message boundaries: one send can\narrive as several recvs, and several sends as one. With chunk=k, sends and recvs move at most\nk bytes at a time (chosen with a seeded random generator, so runs repeat exactly).\n"""\nimport random\nfrom collections import deque\n\n\nclass _Channel:\n    def __init__(self):\n        self.data = deque()\n        self.closed = False\n\n\nclass FakeSocket:\n    def __init__(self, inbox, outbox, chunk, rng, name):\n        self._inbox, self._outbox = inbox, outbox\n        self._chunk, self._rng, self.name = chunk, rng, name\n        self.closed = False\n        self.sent = 0\n\n    def _size(self, n):\n        if self._chunk is None:\n            return n\n        return max(1, min(n, self._rng.randint(1, self._chunk)))\n\n    def send(self, data):\n        if self.closed:\n            raise OSError("send on a closed socket")\n        if self._outbox.closed:\n            raise BrokenPipeError("the other end has closed")\n        data = bytes(data)\n        if not data:\n            return 0\n        n = self._size(len(data))\n        self._outbox.data.extend(data[:n])\n        self.sent += n\n        return n\n\n    def recv(self, bufsize):\n        if self.closed:\n            raise OSError("recv on a closed socket")\n        if bufsize < 1:\n            raise ValueError("bufsize must be positive")\n        if not self._inbox.data:\n            if self._inbox.closed:\n                return b""\n            raise BlockingIOError("no data yet: in a real program this would wait")\n        n = self._size(min(bufsize, len(self._inbox.data)))\n        return bytes(self._inbox.data.popleft() for _ in range(n))\n\n    def shutdown(self):\n        """Finish sending: the other end reads b"" after the data already sent."""\n        self._outbox.closed = True\n\n    def close(self):\n        if not self.closed:\n            self.closed = True\n            self._outbox.closed = True\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self, *exc):\n        self.close()\n\n    def __repr__(self):\n        return f"<FakeSocket {self.name}{\' closed\' if self.closed else \'\'}>"\n\n\ndef socketpair(chunk=None, seed=0):\n    """Two connected FakeSockets. chunk limits how many bytes one send or recv moves."""\n    rng = random.Random(seed)\n    a_to_b, b_to_a = _Channel(), _Channel()\n    return FakeSocket(b_to_a, a_to_b, chunk, rng, "a"), FakeSocket(a_to_b, b_to_a, chunk, rng, "b")\n')

**Predict before you run:** what will this print?



In [ ]:
import json

def handle(method, path):
    if path == "/hello" and method == "GET":
        return 200, {"Content-Type": "text/plain"}, b"Hello!"
    if path == "/hello":
        return 405, {"Allow": "GET"}, b""
    return 404, {"Content-Type": "application/json"}, json.dumps({"error": "not found"}).encode()

for method, path in [("GET", "/hello"), ("POST", "/hello"), ("GET", "/nope")]:
    status, headers, body = handle(method, path)
    print(method, path, "->", status, headers, body)

In [ ]:
import re

class Router:
    def __init__(self):
        self.routes = []

    def add(self, method, pattern, handler):
        def group(m):
            kind, name = m.groups()
            return rf"(?P<{name}>\d+)" if kind == "int" else rf"(?P<{name}>[^/]+)"
        regex = re.sub(r"<(?:(int):)?(\w+)>", group, pattern)
        self.routes.append((method, re.compile(f"^{regex}$"), handler))

    def route(self, method, path):
        allowed = []
        for m, regex, handler in self.routes:
            match = regex.match(path)
            if match:
                if m == method:
                    params = {k: int(v) if v.isdigit() else v for k, v in match.groupdict().items()}
                    return 200, handler, params
                allowed.append(m)
        return (405, sorted(allowed), {}) if allowed else (404, None, {})

router = Router()
router.add("GET", "/books", lambda: "all books")
router.add("GET", "/books/<int:book_id>", lambda book_id: f"book {book_id}")
router.add("DELETE", "/books/<int:book_id>", lambda book_id: f"deleted {book_id}")
for method, path in [("GET", "/books/42"), ("PUT", "/books/42"), ("GET", "/authors")]:
    status, handler, params = router.route(method, path)
    print(method, path, status, handler(**params) if status == 200 else handler)

In [ ]:
import io, json
from wsgiref.util import setup_testing_defaults
from wsgiref.validate import validator

def app(environ, start_response):
    if environ["PATH_INFO"] == "/echo" and environ["REQUEST_METHOD"] == "POST":
        length = int(environ.get("CONTENT_LENGTH") or 0)
        data = json.loads(environ["wsgi.input"].read(length))
        body = json.dumps({"you sent": data, "agent": environ.get("HTTP_USER_AGENT")}).encode()
        start_response("200 OK", [("Content-Type", "application/json"), ("Content-Length", str(len(body)))])
        return [body]
    start_response("404 Not Found", [("Content-Type", "text/plain")])
    return [b"not found"]

body = b'{"n": 7}'
environ = {"REQUEST_METHOD": "POST", "SCRIPT_NAME": "", "PATH_INFO": "/echo", "QUERY_STRING": "", "CONTENT_LENGTH": str(len(body)),
           "CONTENT_TYPE": "application/json", "HTTP_USER_AGENT": "atlas/1.0", "wsgi.input": io.BytesIO(body)}
setup_testing_defaults(environ)
captured = {}
def start_response(status, headers, exc_info=None):
    captured["status"], captured["headers"] = status, headers
    return lambda data: None
result = validator(app)(environ, start_response)
print(captured["status"], b"".join(result))
result.close()

> **Key idea.** A server reads a request (head to the blank line, then Content-Length bytes), routes it, calls the handler, and writes the response with sendall, repeating while the connection is kept alive; it must also handle oversized requests, slow clients and handler errors (500, logged). Routers compile path patterns with parameters into regular expressions; no match is 404, a match with the wrong method is 405 with Allow; put literal routes before overlapping patterns. WSGI applications take environ (REQUEST_METHOD, PATH_INFO, QUERY_STRING, CONTENT_LENGTH, HTTP_* headers, wsgi.input) and start_response(status line, header pairs), and return an iterable of bytes; middleware wraps applications. wsgiref has testing defaults, a validator and a simple server; production uses servers like Gunicorn behind a reverse proxy.

### Exercise: A router with parameters

Write a class `Router` with `add(method, pattern, handler)` and `match(method, path)`. Patterns may contain `<name>` (one path segment, any characters but `/`), `<int:name>` (digits, passed as an `int`) and `<path:name>` (the rest of the path, slashes included); everything else matches literally (escape it with `re.escape`). `match` returns `(handler, params)` for the **first** route (in the order added) whose pattern and method both match; if some route's pattern matches but none with this method, raise `MethodNotAllowed` (given) with the sorted list of allowed methods as its `allowed` attribute; if no pattern matches, raise `NotFound`. A trailing slash on the path is ignored (except for `/` itself). Also a decorator method `route(pattern, methods=("GET",))` that registers the function for each method and returns it unchanged.

In [ ]:
import re

class NotFound(Exception):
    pass

class MethodNotAllowed(Exception):
    def __init__(self, allowed):
        super().__init__(", ".join(allowed))
        self.allowed = allowed

class Router:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_router():\n    \"patterns, parameters, 404 and 405\"\n    r = Router()\n    @r.route(\"/books\")\n    def books():\n        return \"all\"\n    @r.route(\"/books/new\")\n    def new_form():\n        return \"form\"\n    @r.route(\"/books/<int:book_id>\", methods=(\"GET\", \"DELETE\"))\n    def book(book_id):\n        return book_id\n    r.add(\"GET\", \"/users/<name>\", lambda name: name)\n    r.add(\"GET\", \"/files/<path:rest>\", lambda rest: rest)\n    r.add(\"GET\", \"/c++/<name>\", lambda name: name)\n    assert r.match(\"GET\", \"/books\") == (books, {}) and r.match(\"GET\", \"/books/\") == (books, {})\n    assert r.match(\"GET\", \"/books/new\") == (new_form, {}) and r.match(\"DELETE\", \"/books/42\") == (book, {\"book_id\": 42})\n    assert r.match(\"GET\", \"/users/ada\")[1] == {\"name\": \"ada\"} and r.match(\"GET\", \"/files/a/b/c.txt\")[1] == {\"rest\": \"a/b/c.txt\"}\n    assert r.match(\"GET\", \"/c++/x\")[1] == {\"name\": \"x\"}\n    try:\n        r.match(\"PUT\", \"/books/42\")\n        assert False\n    except MethodNotAllowed as e:\n        assert e.allowed == [\"DELETE\", \"GET\"]\n    for path in [\"/books/abc/def\", \"/authors\", \"/users/a/b\", \"/books/x1\"]:\n        try:\n            r.match(\"GET\", path)\n            assert False, path\n        except NotFound:\n            pass")

### Exercise: From request to environ

Write `make_environ(raw, scheme="http")`: parse a raw request (bytes) and build a WSGI `environ`: `REQUEST_METHOD`, `PATH_INFO` (the path, with percent-encoding decoded by `urllib.parse.unquote`), `QUERY_STRING` (as sent, or `""`), `SERVER_PROTOCOL`, `SERVER_NAME` and `SERVER_PORT` (from the `Host` header, the port as a string, defaulting to 80 or 443 by scheme), `CONTENT_TYPE` and `CONTENT_LENGTH` (only if the request has them), `HTTP_...` for every other header (repeated ones joined with `,`), `wsgi.input` (a `BytesIO` of the body), `wsgi.url_scheme`, `wsgi.version` `(1, 0)`, `wsgi.errors` (a `StringIO`), `SCRIPT_NAME` `""`, and `wsgi.multithread`, `wsgi.multiprocess` and `wsgi.run_once` all `False`.

In [ ]:
import io
from urllib.parse import unquote

def make_environ(raw, scheme="http"):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_environ():\n    \"the WSGI environ\"\n    from wsgiref.validate import validator\n    raw = (b\"POST /caf%C3%A9/menu?lang=en&x=1 HTTP/1.1\\r\\nHost: shop.example.com:8080\\r\\nUser-Agent: atlas\\r\\n\"\n           b\"Content-Type: application/json\\r\\nContent-Length: 7\\r\\nX-Tag: a\\r\\nX-Tag: b\\r\\n\\r\\n{\\\"a\\\":1}\")\n    env = make_environ(raw)\n    assert (env[\"REQUEST_METHOD\"], env[\"PATH_INFO\"], env[\"QUERY_STRING\"]) == (\"POST\", \"/caf\u00e9/menu\", \"lang=en&x=1\")\n    assert (env[\"SERVER_NAME\"], env[\"SERVER_PORT\"], env[\"SERVER_PROTOCOL\"]) == (\"shop.example.com\", \"8080\", \"HTTP/1.1\")\n    assert env[\"CONTENT_LENGTH\"] == \"7\" and env[\"CONTENT_TYPE\"] == \"application/json\" and \"HTTP_CONTENT_TYPE\" not in env\n    assert env[\"HTTP_USER_AGENT\"] == \"atlas\" and env[\"HTTP_X_TAG\"] == \"a,b\" and env[\"wsgi.input\"].read() == b'{\"a\":1}'\n    env2 = make_environ(b\"GET / HTTP/1.1\\r\\nHost: a.com\\r\\n\\r\\n\", scheme=\"https\")\n    assert env2[\"SERVER_PORT\"] == \"443\" and env2[\"QUERY_STRING\"] == \"\" and \"CONTENT_LENGTH\" not in env2\n    def app(environ, start_response):\n        start_response(\"200 OK\", [(\"Content-Type\", \"text/plain\")])\n        return [b\"ok\"]\n    result = validator(app)(env2, lambda status, headers, exc_info=None: (lambda data: None))\n    result.close()")

### Exercise: Calling a WSGI application

Write `call_app(app, environ)`: call a WSGI application the way a server does, and return `(status, headers, body)`: the status line text, the list of header pairs, and the body bytes joined from the returned iterable. Pass a `start_response` that records its arguments and returns a `write` function (the old, rarely used way to send body data, which you should also collect, **before** the iterable's data). If the returned iterable has a `close` method, call it at the end, even if iterating raised. If `start_response` was never called, raise `RuntimeError`.

In [ ]:
def call_app(app, environ):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_callapp():\n    \"the server's side of WSGI\"\n    import io\n    def hello(environ, start_response):\n        start_response(\"200 OK\", [(\"Content-Type\", \"text/plain\")])\n        return [b\"Hello, \", environ[\"PATH_INFO\"][1:].encode(), b\"!\"]\n    assert call_app(hello, {\"PATH_INFO\": \"/Ada\"}) == (\"200 OK\", [(\"Content-Type\", \"text/plain\")], b\"Hello, Ada!\")\n    closed = []\n    class Body:\n        def __iter__(self):\n            yield b\"one \"\n            yield b\"two\"\n        def close(self):\n            closed.append(True)\n    def streaming(environ, start_response):\n        write = start_response(\"201 Created\", [])\n        write(b\"first: \")\n        return Body()\n    assert call_app(streaming, {}) == (\"201 Created\", [], b\"first: one two\") and closed == [True]\n    def broken(environ, start_response):\n        return [b\"no status\"]\n    try:\n        call_app(broken, {})\n        assert False\n    except RuntimeError:\n        pass")

### Exercise: A JSON application

Write a WSGI application `books_app` (using `BOOKS`, a module-level dict of id to book, which the app may change) with these routes: `GET /books` (all books, as a JSON list, by id), `GET /books/<id>` (one book, or `404` with `{"error": "not found"}`), `POST /books` (a JSON body with `title` and `author`, both non-empty strings; `201` with the new book, ids counting up from the largest, and a `Location: /books/ID` header; `400` with `{"error": "invalid JSON"}` for a bad body, `422` with `{"error": "title and author are required"}` for missing fields) and `DELETE /books/<id>` (`204` with no body, or `404`). Any other method on a known path is `405` with an `Allow` header (methods sorted, comma-separated); any other path `404`. Every response with a body is JSON with `Content-Type: application/json` and a `Content-Length`.

In [ ]:
import json, re

BOOKS = {1: {"id": 1, "title": "Emma", "author": "Austen"}, 2: {"id": 2, "title": "Dune", "author": "Herbert"}}

def books_app(environ, start_response):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_app():\n    \"routes, statuses and JSON\"\n    import io, json\n    from wsgiref.util import setup_testing_defaults\n    def call(method, path, body=b\"\"):\n        env = {\"REQUEST_METHOD\": method, \"PATH_INFO\": path, \"CONTENT_LENGTH\": str(len(body)), \"wsgi.input\": io.BytesIO(body)}\n        setup_testing_defaults(env)\n        got = {}\n        def start_response(status, headers, exc_info=None):\n            got[\"status\"], got[\"headers\"] = status, dict(headers)\n        out = b\"\".join(books_app(env, start_response))\n        return got[\"status\"], got[\"headers\"], out\n    status, headers, out = call(\"GET\", \"/books\")\n    assert status == \"200 OK\" and headers[\"Content-Type\"] == \"application/json\" and [b[\"id\"] for b in json.loads(out)] == [1, 2]\n    assert headers[\"Content-Length\"] == str(len(out))\n    assert json.loads(call(\"GET\", \"/books/2\")[2])[\"title\"] == \"Dune\" and call(\"GET\", \"/books/9\")[0] == \"404 Not Found\"\n    status, headers, out = call(\"POST\", \"/books\", b'{\"title\": \"Ivanhoe\", \"author\": \"Scott\"}')\n    assert status == \"201 Created\" and headers[\"Location\"] == \"/books/3\" and json.loads(out)[\"id\"] == 3\n    assert call(\"POST\", \"/books\", b\"{nope\")[0] == \"400 Bad Request\" and call(\"POST\", \"/books\", b'{\"title\": \"\"}')[0].startswith(\"422\")\n    status, headers, out = call(\"DELETE\", \"/books/1\")\n    assert status == \"204 No Content\" and out == b\"\" and call(\"DELETE\", \"/books/1\")[0] == \"404 Not Found\"\n    status, headers, _ = call(\"PUT\", \"/books/2\")\n    assert status == \"405 Method Not Allowed\" and headers[\"Allow\"] == \"DELETE, GET\"\n    assert call(\"GET\", \"/authors\")[0] == \"404 Not Found\"")

### Exercise: Serving files safely

Write `resolve(root, url_path)`: the file inside the folder `root` (a `pathlib.Path`) that a URL path names, or `None` if it would escape `root` or doesn't exist as a file. Decode percent-encoding first (so `%2e%2e` is caught), refuse anything containing a NUL character, serve `index.html` for a folder, and check the resolved path is inside the resolved root with `Path.is_relative_to`. Then `content_type(path)`: from `mimetypes.guess_type`, defaulting to `application/octet-stream`, with `; charset=utf-8` added to `text/...` types.

In [ ]:
import mimetypes
from pathlib import Path
from urllib.parse import unquote

def resolve(root, url_path):
    ___

def content_type(path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_static():\n    \"no escaping the folder\"\n    import os\n    from pathlib import Path\n    root = Path(\"site\")\n    (root / \"css\").mkdir(parents=True, exist_ok=True)\n    (root / \"index.html\").write_text(\"<h1>home</h1>\")\n    (root / \"css\" / \"main.css\").write_text(\"body{}\")\n    Path(\"secret.txt\").write_text(\"keep out\")\n    assert resolve(root, \"/\") == (root / \"index.html\").resolve() and resolve(root, \"/css/main.css\") == (root / \"css\" / \"main.css\").resolve()\n    for bad in [\"/../secret.txt\", \"/css/../../secret.txt\", \"/%2e%2e/secret.txt\", \"/nope.html\", \"/css\", \"/a%00b\"]:\n        assert resolve(root, bad) is None, bad\n    assert content_type(Path(\"a.css\")) == \"text/css; charset=utf-8\" and content_type(Path(\"a.png\")) == \"image/png\"\n    assert content_type(Path(\"x.unknownthing\")) == \"application/octet-stream\"")

### Exercise: Middleware

Write two WSGI middleware functions. `log_requests(app, log)` returns an application that calls `app` and appends `"METHOD PATH STATUS"` (the status code only, such as `GET /books 200`) to the list `log` once the response has started. `security_headers(app)` returns an application that adds `X-Content-Type-Options: nosniff` and `X-Frame-Options: DENY` to every response's headers, unless the app already set them (in any case). They must compose: `log_requests(security_headers(app), log)`.

In [ ]:
def log_requests(app, log):
    ___

def security_headers(app):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_middleware():\n    \"wrapping applications\"\n    def app(environ, start_response):\n        if environ[\"PATH_INFO\"] == \"/frame\":\n            start_response(\"200 OK\", [(\"x-frame-options\", \"SAMEORIGIN\")])\n        else:\n            start_response(\"404 Not Found\", [(\"Content-Type\", \"text/plain\")])\n        return [b\"body\"]\n    log = []\n    wrapped = log_requests(security_headers(app), log)\n    seen = {}\n    def start_response(status, headers, exc_info=None):\n        seen[\"status\"], seen[\"headers\"] = status, headers\n    assert b\"\".join(wrapped({\"REQUEST_METHOD\": \"GET\", \"PATH_INFO\": \"/x\"}, start_response)) == b\"body\"\n    names = [h[0].lower() for h in seen[\"headers\"]]\n    assert \"x-content-type-options\" in names and \"x-frame-options\" in names and log == [\"GET /x 404\"]\n    wrapped({\"REQUEST_METHOD\": \"POST\", \"PATH_INFO\": \"/frame\"}, start_response)\n    assert [h for h in seen[\"headers\"] if h[0].lower() == \"x-frame-options\"] == [(\"x-frame-options\", \"SAMEORIGIN\")]\n    assert log == [\"GET /x 404\", \"POST /frame 200\"]")

### Bug hunt: A body that was a string

A WSGI application must return an iterable of **bytes**, but this one returns a `str`, which iterates as single characters, and joining them as bytes fails with `TypeError`. Return a list with one encoded bytes object, so this prints `200 OK b'Hello, Ada!'`.

In [ ]:
def app(environ, start_response):
    start_response("200 OK", [("Content-Type", "text/plain; charset=utf-8")])
    return "Hello, Ada!"

got = []
body = b"".join(app({}, lambda status, headers: got.append(status)))
print(got[0], body)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"an iterable of bytes\"\n    expect_output(\"200 OK b'Hello, Ada!'\")")

### Bug hunt: A route that could never match

`/books/<name>` was registered before `/books/new`, and the first matching route wins, so `/books/new` goes to the book handler and this prints `book called new`. Register the literal route first, so it prints `the new-book form`.

In [ ]:
import re

routes = []
def add(pattern, handler):
    regex = re.sub(r"<(\w+)>", r"(?P<\1>[^/]+)", pattern)
    routes.append((re.compile(f"^{regex}$"), handler))

add("/books/<name>", lambda name: f"book called {name}")
add("/books/new", lambda: "the new-book form")

def dispatch(path):
    for regex, handler in routes:
        m = regex.match(path)
        if m:
            return handler(**m.groupdict())

print(dispatch("/books/new"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"literal routes first\"\n    expect_output(\"the new-book form\")")

### Bug hunt: A path that escaped

The file server joins the URL path onto its folder and checks only that the result **starts with** the folder's name as text, so `/../site-private/key.txt` (a folder whose name merely begins with `site`) passes the check and this prints `served: shh`. Compare resolved paths with `is_relative_to`, so it prints `refused`.

In [ ]:
from pathlib import Path

Path("site").mkdir(exist_ok=True)
Path("site-private").mkdir(exist_ok=True)
Path("site-private/key.txt").write_text("shh")

def serve(url_path):
    root = Path("site").resolve()
    target = (root / url_path.lstrip("/")).resolve()
    if str(target).startswith(str(root)) and target.is_file():
        return "served: " + target.read_text()
    return "refused"

print(serve("/../site-private/key.txt"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a real containment check\"\n    expect_output(\"refused\")")

### Puzzle: A WSGI application

Arrange the lines of a WSGI application that greets the name in the query string. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the status for a path that exists but doesn't accept this method
q = ___   # the header that 405 response should carry
r = ___   # the environ key holding the request's User-Agent header
s = ___   # True or False: a WSGI application may return a str
t = ___   # the regular expression group "<int:book_id>" becomes, as a string

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 405 and q == \"Allow\" and r == \"HTTP_USER_AGENT\", \"p, q and r\"\n    assert s is False and t in (r\"(?P<book_id>\\d+)\", \"(?P<book_id>\\\\d+)\"), \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: serving a connection

Write `serve_connection(sock, app, max_body=1_000_000)`: serve HTTP/1.1 requests arriving on one socket (a `netsim` socket in the checks) to a WSGI application, one after another, until the client closes the connection or sends `Connection: close`. For each request: read the head (to the blank line) and its `Content-Length` body, buffering across `recv`s (and keeping any extra bytes for the next request); build the environ with `make_environ` (given); call the app with `call_app` (given); and send `HTTP/1.1 STATUS` plus the app's headers, adding `Content-Length` if the app didn't, plus `Connection: close` on the last response when the client asked for it, then the body, all of it. A body over `max_body` gets `413 Payload Too Large` (without calling the app), and an app that raises gets a `500 Internal Server Error` with the body `internal error` (the server keeps serving). Return the number of requests answered.

In [ ]:
import io
from urllib.parse import unquote

def make_environ(raw, scheme="http"):
    head, _, rest = raw.partition(b"\r\n\r\n")
    lines = head.decode("latin-1").split("\r\n")
    method, target, protocol = lines[0].split(" ")
    path, _, query = target.partition("?")
    environ = {"REQUEST_METHOD": method, "SCRIPT_NAME": "", "PATH_INFO": unquote(path), "QUERY_STRING": query,
               "SERVER_PROTOCOL": protocol, "SERVER_NAME": "localhost", "SERVER_PORT": "80", "wsgi.version": (1, 0),
               "wsgi.url_scheme": scheme, "wsgi.errors": io.StringIO(), "wsgi.multithread": False,
               "wsgi.multiprocess": False, "wsgi.run_once": False}
    headers = {}
    for line in lines[1:]:
        name, _, value = line.partition(":")
        headers[name.strip().upper().replace("-", "_")] = value.strip()
    length = int(headers.get("CONTENT_LENGTH", "0") or 0)
    environ["wsgi.input"] = io.BytesIO(rest[:length])
    for key, value in headers.items():
        environ[key if key in ("CONTENT_TYPE", "CONTENT_LENGTH") else f"HTTP_{key}"] = value
    return environ

def call_app(app, environ):
    captured, written = {}, []
    def start_response(status, headers, exc_info=None):
        captured["status"], captured["headers"] = status, list(headers)
        return written.append
    result = app(environ, start_response)
    try:
        chunks = [bytes(c) for c in result]
    finally:
        if hasattr(result, "close"):
            result.close()
    return captured["status"], captured["headers"], b"".join(written) + b"".join(chunks)

def serve_connection(sock, app, max_body=1_000_000):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_serve():\n    \"keep-alive, framing, errors and limits\"\n    from netsim import socketpair\n    def app(environ, start_response):\n        if environ[\"PATH_INFO\"] == \"/boom\":\n            raise RuntimeError(\"bug\")\n        length = int(environ.get(\"CONTENT_LENGTH\") or 0)\n        data = environ[\"wsgi.input\"].read(length)\n        body = f\"{environ['REQUEST_METHOD']} {environ['PATH_INFO']} {len(data)}\".encode()\n        start_response(\"200 OK\", [(\"Content-Type\", \"text/plain\")])\n        return [body]\n    client, server = socketpair(chunk=7, seed=3)\n    requests = (b\"GET /a HTTP/1.1\\r\\nHost: x\\r\\n\\r\\n\"\n                b\"POST /b HTTP/1.1\\r\\nHost: x\\r\\nContent-Length: 5\\r\\n\\r\\nhello\"\n                b\"GET /boom HTTP/1.1\\r\\nHost: x\\r\\n\\r\\n\"\n                b\"POST /big HTTP/1.1\\r\\nHost: x\\r\\nContent-Length: 50\\r\\n\\r\\n\" + b\"x\" * 50 +\n                b\"GET /last HTTP/1.1\\r\\nHost: x\\r\\nConnection: close\\r\\n\\r\\n\"\n                b\"GET /ignored HTTP/1.1\\r\\nHost: x\\r\\n\\r\\n\")\n    while requests:\n        requests = requests[client.send(requests):]\n    assert serve_connection(server, app, max_body=20) == 5\n    replies = b\"\"\n    while True:\n        try:\n            chunk = client.recv(4096)\n        except BlockingIOError:\n            break\n        if not chunk:\n            break\n        replies += chunk\n    parts = replies.split(b\"HTTP/1.1 \")[1:]\n    assert [p.split(b\"\\r\\n\")[0] for p in parts] == [b\"200 OK\", b\"200 OK\", b\"500 Internal Server Error\", b\"413 Payload Too Large\", b\"200 OK\"]\n    assert parts[0].endswith(b\"\\r\\n\\r\\nGET /a 0\") and parts[1].endswith(b\"POST /b 5\") and parts[2].endswith(b\"internal error\")\n    assert b\"Content-Length: 8\" in parts[0] and b\"Connection: close\" in parts[4]\n    c2, s2 = socketpair()\n    c2.send(b\"GET / HTTP/1.1\\r\\nHost: x\\r\\n\\r\\n\")\n    c2.shutdown()\n    assert serve_connection(s2, app) == 1")

## Warm-up

At heart, a web application is a **function from a request to a response**. Everything else in a server (sockets, parsing, threads, TLS) exists to call that function for each request that arrives and send back what it returns. Keeping that function separate from the machinery is what lets the same application run on a laptop, in a test, and on a production server.

## A server in four steps

For each connection, an HTTP server repeats:

1. **Read a request** from the socket: bytes up to the blank line (`\r\n\r\n`), then `Content-Length` bytes of body (Day 2), buffering whatever `recv` returns (Day 4).
2. **Route** it: find the handler for this method and path.
3. **Call the handler**, which returns a status, headers and a body.
4. **Write the response**, with a `Content-Length`, using `sendall`; then, if the connection is kept alive, go back to step 1 for the next request; otherwise close.

Servers also have to be robust in ways a tidy example isn't: a request too large to accept (`413`), a client that sends half a request and stops (a timeout), a handler that raises (a `500`, with the details logged rather than shown to the user), and many clients at once.

## Routing

A **router** maps `(method, path)` to a handler. Real paths carry data: `/books/42` means book 42, so routes are **patterns** with **parameters**: `/books/<int:book_id>` matches `/books/42` and passes `book_id=42`.

- Compile each pattern to a regular expression once: `<int:name>` becomes `(?P<name>\d+)`, `<name>` becomes `(?P<name>[^/]+)` (any characters except `/`), and everything else is matched literally.
- If no pattern matches the path, answer **`404 Not Found`**. If the path matches but not with this method, answer **`405 Method Not Allowed`** with an **`Allow`** header listing the methods that would work: that's the difference between "no such thing" and "you can't do that to it".
- Order matters when patterns overlap: `/books/new` must be tried before `/books/<name>`, or the literal route can never match.

Frameworks register routes with decorators, which you know how to build (Week 60): `@app.route("/books/<int:book_id>", methods=["GET"])`.

## WSGI

If every framework talked to every server in its own way, you couldn't run Django on a server written for Flask. **WSGI** (the Web Server Gateway Interface, PEP 3333) is the agreement between them. A WSGI **application** is any callable:

```python
def app(environ, start_response):
    start_response("200 OK", [("Content-Type", "text/plain; charset=utf-8")])
    return [b"Hello from WSGI\n"]
```

- **`environ`** is a dict describing the request: `REQUEST_METHOD`, `PATH_INFO` (the path), `QUERY_STRING`, `CONTENT_TYPE`, `CONTENT_LENGTH`, each request header as `HTTP_NAME` (upper case, `-` replaced by `_`: `HTTP_USER_AGENT`), `SERVER_NAME` and `SERVER_PORT`, `wsgi.url_scheme` (`http` or `https`), and **`wsgi.input`**, a file-like object to read the body from.
- **`start_response(status, headers)`** is called once with the status line text (`"404 Not Found"`, code **and** reason) and a list of `(name, value)` header pairs.
- The application **returns an iterable of bytes** (a list with one item is fine): the body. Not a string; bytes.

The server builds `environ`, calls the application, and writes what it gets back. **Middleware** is a WSGI application that wraps another, adding behaviour on the way in or out: logging every request, adding security headers, compressing responses, checking authentication. Because both sides have the same interface, middleware stacks like decorators.

The standard library's `wsgiref` has the pieces: `wsgiref.util.setup_testing_defaults(environ)` fills in a test environ, `wsgiref.validate.validator(app)` checks an application follows the rules, and `wsgiref.simple_server` is a small real server. (For `async` applications there's an equivalent interface, **ASGI**, which FastAPI uses: next week.)

## The real servers

On your own computer, Python can serve straight away:

```bash
python -m http.server 8000
```

serves the current folder's files at `http://127.0.0.1:8000/` (for development only). To run a WSGI application:

```python
from wsgiref.simple_server import make_server

with make_server("127.0.0.1", 8000, app) as server:
    server.serve_forever()
```

For production, a WSGI application runs under a server built for it, such as **Gunicorn** (`gunicorn myapp:app --workers 4`), usually behind a **reverse proxy** such as nginx that handles TLS, static files and slow clients (Week 77). `http.server.BaseHTTPRequestHandler` also lets you write handlers directly (`do_GET`, `do_POST`), which is fine for small tools.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, run `books_app` with `wsgiref.simple_server.make_server`, and call it with `curl` from yesterday and with your Day 3 client. Tomorrow's Forge builds the whole stack: a small web framework with routing, request and response objects, middleware and a test client.